# Réseau de cooccurrences entre personnages

Ce notebook utilise :

- `../Occurences/pern_aliases_utilises.csv`
- `../Texte/La-ballade-de-Pern-intégrale-sentences-lemma.csv`
- `../Noms personnages.ods`

Objectifs :

1. calculer, pour chaque personnage, le nombre de phrases où il/elle apparaît **seul·e** ;
2. calculer le nombre de cooccurrences de personnages dans une même phrase ;
3. produire un **résumé de genre** en 5 cases :
   - hommes seuls
   - femmes seules
   - hommes avec femmes
   - hommes avec hommes
   - femmes avec femmes


## Remarque méthodologique

Les alias **ambigus** (un même alias rattaché à plusieurs IDs) sont exclus par défaut pour éviter les faux positifs.  
La liste est sauvegardée dans `aliases_ambigus_exclus.csv`.


In [3]:

from pathlib import Path
import itertools
import re
import unicodedata

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
ALIASES_FILE = BASE_DIR.parent / "Occurences" / "pern_aliases_utilises.csv"
TEXT_FILE = BASE_DIR.parent / "Texte" / "La-ballade-de-Pern-intégrale-sentences-lemma.csv"
CHAR_FILE = BASE_DIR.parent / "Noms personnages.ods"

OUTPUT_GENDER = BASE_DIR / "resume_genre_personnages.csv"
OUTPUT_SENT_MATCH = BASE_DIR / "phrases_personnages_detectes.csv"
OUTPUT_AMBIG = BASE_DIR / "aliases_ambigus_exclus.csv"

def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value)
    value = unicodedata.normalize("NFKC", value)
    value = value.replace("’", "'").replace("‘", "'").replace("`", "'").replace("´", "'")
    value = re.sub(r"\s+", " ", value)
    return value.strip()

aliases_df = pd.read_csv(ALIASES_FILE)
text_df = pd.read_csv(TEXT_FILE, usecols=["sentence_id", "book", "text"])
chars_df = pd.read_excel(CHAR_FILE, sheet_name="Noms et genres persos", engine="odf")

meta = chars_df[["ID unique", "Noms humains", "H/F/Na"]].copy()
meta["ID unique"] = meta["ID unique"].astype(int)

print(f"Nombre de phrases : {len(text_df):,}".replace(",", " "))
print(f"Nombre de personnages : {len(meta):,}".replace(",", " "))


/home/marc-at/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/marc-at/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


Nombre de phrases : 132 850
Nombre de personnages : 1 026


In [4]:

# Construire la table des alias
alias_map = {}

for _, row in aliases_df[["ID unique", "aliases"]].iterrows():
    cid = int(row["ID unique"])
    alias_field = normalize_text(row["aliases"])
    for part in [p.strip() for p in alias_field.split("|") if str(p).strip()]:
        alias_key = normalize_text(part).lower()
        alias_map.setdefault(alias_key, set()).add(cid)

# Ajouter le nom principal si jamais il n'est pas déjà présent
for _, row in meta.iterrows():
    canonical = normalize_text(row["Noms humains"]).lower()
    if canonical and canonical not in alias_map:
        alias_map[canonical] = {int(row["ID unique"])}

unique_alias_map = {alias: next(iter(ids)) for alias, ids in alias_map.items() if len(ids) == 1}
ambiguous_aliases = sorted(alias for alias, ids in alias_map.items() if len(ids) > 1)

pd.DataFrame({"alias_ambigu_exclu": ambiguous_aliases}).to_csv(OUTPUT_AMBIG, index=False)

print(f"Alias totaux : {len(alias_map):,}".replace(",", " "))
print(f"Alias uniques conservés : {len(unique_alias_map):,}".replace(",", " "))
print(f"Alias ambigus exclus : {len(ambiguous_aliases):,}".replace(",", " "))


Alias totaux : 1 414
Alias uniques conservés : 1 386
Alias ambigus exclus : 28


In [5]:

# Regex global pour détecter les alias dans les phrases
escaped_aliases = sorted((re.escape(alias) for alias in unique_alias_map.keys()), key=len, reverse=True)
pattern = re.compile(
    r"(?<!\w)(" + "|".join(escaped_aliases) + r")(?!\w)",
    flags=re.IGNORECASE
)

records = []

for sentence_id, book, text in text_df.itertuples(index=False):
    clean_text = normalize_text(text).lower()
    char_ids = {unique_alias_map[m.group(1).lower()] for m in pattern.finditer(clean_text)}
    if char_ids:
        records.append((sentence_id, book, tuple(sorted(char_ids))))

sent_matches = pd.DataFrame(records, columns=["sentence_id", "book", "character_ids"])
sent_matches["n_characters"] = sent_matches["character_ids"].apply(len)

print(f"Phrases avec au moins un personnage détecté : {len(sent_matches):,}".replace(",", " "))
sent_matches.head()


Phrases avec au moins un personnage détecté : 48 191


,sentence_id,book,character_ids,n_characters
0,8,NaN,"(718,)",1
1,21,1.0,"(718,)",1
2,22,1.0,"(262,)",1
3,23,1.0,"(262,)",1
4,31,1.0,"(274, 718)",2


In [6]:

# Comptes solo
solo = sent_matches[sent_matches["n_characters"] == 1].copy()
solo["ID unique"] = solo["character_ids"].str[0]
solo_counts = solo.groupby("ID unique").size().rename("solo_sentence_count")

# Comptes de présence en cooccurrence (le personnage apparaît dans une phrase avec au moins un autre)
multi = sent_matches[sent_matches["n_characters"] >= 2].copy()
multi_exp = multi.explode("character_ids").rename(columns={"character_ids": "ID unique"})
co_sentence_counts = multi_exp.groupby("ID unique").size().rename("cooccurs_in_sentence_count")


In [9]:
# Résumé par genre
gender_map = meta.set_index("ID unique")["H/F/Na"].to_dict()

# 1) Cas "seul dans une phrase"
solo_gender = solo["ID unique"].map(gender_map)

hommes_seuls = int((solo_gender == "HOMME").sum())
femmes_seules = int((solo_gender == "FEMME").sum())

# 2) Cas "avec quelqu'un" dans les cooccurrences
hommes_avec_hommes = 0
femmes_avec_femmes = 0
hommes_avec_femmes = 0

for a, b, w in edges[["a", "b", "w"]].itertuples(index=False):
    ga = gender_map.get(a)
    gb = gender_map.get(b)

    # on ignore les cas où le genre n'est pas HOMME/FEMME
    if ga not in {"HOMME", "FEMME"} or gb not in {"HOMME", "FEMME"}:
        continue

    if ga == "HOMME" and gb == "HOMME":
        hommes_avec_hommes += int(w)
    elif ga == "FEMME" and gb == "FEMME":
        femmes_avec_femmes += int(w)
    else:
        hommes_avec_femmes += int(w)

# 3) Tableau final
gender_summary_df = pd.DataFrame({
    "categorie": [
        "homme_seul",
        "femme_seule",
        "homme_avec_homme",
        "femme_avec_femme",
        "homme_avec_femme"
    ],
    "n": [
        hommes_seuls,
        femmes_seules,
        hommes_avec_hommes,
        femmes_avec_femmes,
        hommes_avec_femmes
    ]
})

gender_summary_df.to_csv(OUTPUT_GENDER, index=False)
gender_summary_df

,categorie,n
0,homme_seul,27132
1,femme_seule,10132
2,homme_avec_homme,9380
3,femme_avec_femme,1527
4,homme_avec_femme,6188
